## Understand what 'change' in 10-K/Q means

In [28]:
import sys

sys.path.append("..")
from src.config import DATA_DIR

text_12 = (DATA_DIR / "MSFT" / "2025-12-31_mdna.txt").read_text(encoding="utf-8")
text_03 = (DATA_DIR / "MSFT" / "2026-03-31_mdna.txt").read_text(encoding="utf-8")

In [31]:
# we split the MD&A by paragraphs, to identify which convery 'change'
def split_paragraphs(text:str):
    # paragraphs are split on blank lines ("\n\n")
    split_list = [para.strip() for para in text.split(sep = "\n\n")]

    return split_list

split_12 = split_paragraphs(text_12)
split_03 = split_paragraphs(text_03)

In [32]:
import statistics

print("Statistics for 12: ")

sorted_split_12 = sorted(split_12, key=len)

# 0. Paragraph count
print(len(sorted_split_12))

# 1. Shortest string metrics
shortest_str = sorted_split_12[0]
print(len(shortest_str))

# 2. Longest string metrics
longest_str = sorted_split_12[-1]
print(len(longest_str))

# 3. Median string metrics
median = statistics.median([len(p) for p in sorted_split_12])
print(median)

# 30 shortest paragraphs
print(sorted_split_12[:30])

Statistics for 12: 
268
2
1512
87.0
['30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', 'PART I', 'Item 2', 'PART I', 'Item 2', 'PART I', 'Item 2', 'PART I', 'Item 2', 'PART I', 'Item 2', 'PART I', 'Item 2', 'PART I', 'Item 2']


### Make the above into a function

In [33]:
import statistics

def describe_paragraphs(paras, label="", n_shortest=10):
    """Print basic length statistics for a list of paragraphs."""
    
    lengths = [len(p) for p in paras]
    print(f"--- {label}")
    print(f"count:  {len(paras)}")
    print(f"min:    {min(lengths)}")
    print(f"median: {statistics.median(lengths)}")
    print(f"max:    {max(lengths)}")
    print(f"total:  {sum(lengths):,} chars")
    print(f"shortest {n_shortest}: {sorted(paras, key=len)[:n_shortest]}")

    return None

describe_paragraphs(split_12, "MSFT 2025-12-31 raw")
describe_paragraphs(split_03, "MSFT 2026-03-31 raw")

--- MSFT 2025-12-31 raw
count:  268
min:    2
median: 87.0
max:    1512
total:  56,829 chars
shortest 10: ['30', '31', '32', '33', '34', '35', '36', '37', '38', '39']
--- MSFT 2026-03-31 raw
count:  273
min:    2
median: 84
max:    1512
total:  58,486 chars
shortest 10: ['31', '32', '33', '34', '35', '36', '37', '38', '39', '40']


In [ ]:
# drop paragraphs that are not really relevant to us at all
from collections import Counter

def clean_paragraphs(paras:list):

    para_counter = Counter(paras)
    cleaned_paras = []
    for p in paras:
        if len(p) == 0 or p.isdigit() or para_counter[p] >=3:
            continue
        cleaned_paras.append(p)

    return cleaned_paras

cleaned_split_12 = clean_paragraphs(split_12)
cleaned_split_03 = clean_paragraphs(split_03)

describe_paragraphs(cleaned_split_12, "MSFT 2025-12-31 clean")
describe_paragraphs(cleaned_split_03, "MSFT 2026-03-31 clean")

--- MSFT 2025-12-31 clean
count:  209
min:    7
median: 134
max:    1512
total:  55,591 chars
shortest 10: ['Metrics', 'OVERVIEW', 'Goodwill', 'Valuation', 'Dividends', 'Item 3, 4', 'Commercial', 'Cash Flows', 'Seasonality', 'INCOME TAXES']
--- MSFT 2026-03-31 clean
count:  211
min:    7
median: 187
max:    1512
total:  57,294 chars
shortest 10: ['Metrics', 'OVERVIEW', 'Goodwill', 'Valuation', 'Dividends', 'Item 3, 4', 'Commercial', 'Cash Flows', 'Seasonality', 'INCOME TAXES']


### Find the actual difference, using difflib

#### what this aims to do is, if some paragraph is 100% same, it isn't a 'change' at all - so we can neglect it. 
#### Then if a paragraph is a replacement, i.e. is it saying revenue was 20% vs revenue was 21%, over quarters, this isn't really a significant change. So, we can basically neglect such paragraphs

In [36]:
from difflib import SequenceMatcher
from collections import Counter

old, new = cleaned_split_12, cleaned_split_03   # Dec 2025 -> Mar 2026

sm = SequenceMatcher(None, old, new, autojunk=False)
opcodes = sm.get_opcodes()

# 1. Collect paragraphs by change type
replaced, deleted, inserted = [], [], []
for tag, i1, i2, j1, j2 in opcodes:
    if tag == "replace":
        replaced.append((old[i1:i2], new[j1:j2]))
    elif tag == "delete":
        deleted.extend(old[i1:i2])
    elif tag == "insert":
        inserted.extend(new[j1:j2])

# 2. Paragraphs that were deleted in one place and inserted elsewhere are moves, not changes
moved = set(deleted) & set(inserted)
deleted = [p for p in deleted if p not in moved]
inserted = [p for p in inserted if p not in moved]

# 3. Counts (in paragraphs)
counts = Counter()
for tag, i1, i2, j1, j2 in opcodes:
    counts[tag] += max(i2 - i1, j2 - j1)
print("Raw opcode counts (paragraphs):", dict(counts))
print(f"Replaced blocks: {len(replaced)}  |  Deleted: {len(deleted)}  |  Inserted: {len(inserted)}  |  Moved: {len(moved)}")
print(f"Unchanged share: {counts['equal'] / max(len(old), len(new)):.0%}")

# 4. Look at examples
def show(title, items, n=5):
    print("\n" + "=" * 80 + f"\n{title} (showing {min(n, len(items))} of {len(items)})\n" + "=" * 80)
    for item in items[:n]:
        if isinstance(item, tuple):          # replace: (old paragraphs, new paragraphs)
            print("OLD:", " | ".join(item[0]))
            print("NEW:", " | ".join(item[1]))
        else:
            print(item)
        print("-" * 80)

show("REPLACED", replaced)
show("INSERTED (new this quarter)", inserted)
show("DELETED (gone this quarter)", deleted)

Raw opcode counts (paragraphs): {'equal': 100, 'replace': 137, 'insert': 11}
Replaced blocks: 29  |  Deleted: 0  |  Inserted: 11  |  Moved: 0
Unchanged share: 47%

REPLACED (showing 5 of 29)
OLD: Highlights from the second quarter of fiscal year 2026 compared with the second quarter of fiscal year 2025 included: | •Microsoft Cloud revenue increased 26% to $51.5 billion. | •Commercial remaining performance obligation increased 110% to $625 billion. | •Microsoft 365 Commercial cloud revenue increased 17%. | •Microsoft 365 Consumer cloud revenue increased 29%. | •LinkedIn revenue increased 11%. | •Dynamics 365 revenue increased 19%. | •Azure and other cloud services revenue increased 39%. | •Windows OEM and Devices revenue increased 1%.
NEW: Highlights from the third quarter of fiscal year 2026 compared with the third quarter of fiscal year 2025 included: | •Microsoft Cloud revenue increased 29% to $54.5 billion. | •Commercial remaining performance obligation increased 99% to $627 billion